# REPA × ImageNet-256 × DINOv2 (Colab)

Trains **SiT-S/8** on ImageNet-256 using **DINOv2 ViT-Base** as the representation encoder.

FID Score = 188.4445 (80000 samples, 400k steps)



## 1. Mount Google Drive

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install dependencies

In [14]:
%%bash
pip install -q accelerate diffusers timm einops kaggle

In [15]:
  import os
  path = '/content/drive/MyDrive/last.pt'
  print('Exists:', os.path.isfile(path))
  print('Size:', os.path.getsize(path) / 1e6, 'MB')

Exists: True
Size: 2733.044564 MB


## 3. Download ImageNet-256 via Kaggle API

In [16]:
import os, shutil

# copy kaggle.json from Drive
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# download ImageNet-256
if not os.path.isdir('/content/imagenet256_raw'):
    os.makedirs('/content/imagenet256_raw', exist_ok=True)
    os.system('kaggle datasets download -d dimensi0n/imagenet-256 -p /content/imagenet256_raw --unzip')

print(f'ImageNet-256 exists: {os.path.isdir("/content/imagenet256_raw")}')
print(f'Classes found: {len(os.listdir("/content/imagenet256_raw"))}')

ImageNet-256 exists: True
Classes found: 1000


## 4. Clone REPA fork

In [17]:
%%bash
if [ ! -d "/content/REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git /content/REPA
fi
echo "REPA ready. Commit: $(git -C /content/REPA rev-parse --short HEAD)"

REPA ready. Commit: f127a4a


Cloning into '/content/REPA'...


## 5. Set paths

In [18]:
import os

REPA_DIR      = '/content/REPA'
DATA_DIR      = '/content/data/imagenet256'
IMAGENET_ROOT = '/content/imagenet256_raw'

os.makedirs(DATA_DIR, exist_ok=True)

print(f'REPA dir     : {REPA_DIR}')
print(f'Data dir     : {DATA_DIR}')
print(f'ImageNet dir : {os.path.isdir(IMAGENET_ROOT)}')

REPA dir     : /content/REPA
Data dir     : /content/data/imagenet256
ImageNet dir : True


## 6. Prepare ImageNet-256
VAE-encodes images and saves latents as `.npy`.

In [19]:
import subprocess, sys, threading

cmd = [
    sys.executable, '/content/REPA/prepare_imagenet256.py',
    '--out-dir',           '/content/data/imagenet256',
    '--imagenet-root',     '/content/imagenet256_raw',
    '--resolution',        '256',
    '--max-train-samples', '5000',
]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t1 = threading.Thread(target=stream, args=(process.stdout,))
t2 = threading.Thread(target=stream, args=(process.stderr,))
t1.start(); t2.start()
t1.join(); t2.join()
process.wait()
print('Exit code:', process.returncode)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Using device: cuda
Detected layout: flat class folders — creating 90/10 split

Train: 5000 | Val: 5000
Loading VAE...
train: 100%|██████████| 157/157 [06:09<00:00,  2.35s/it]
[train] saved 5000 images and VAE latents to /content/data/imagenet256

val: 100%|██████████| 157/157 [06:34<00:00,  2.51s/it]
[val] saved 5000 images and VAE latents to /content/data/imagenet256
Exit code: 0


## 7. Train
Runs 30000 steps. Checkpoints saved to `/content/results/`.

**Models**
1. SiT-S/8 (Diffusion Model): denoises in latent space
2. DINOv2 ViT-Base: pretrained vision encoder (downloads automatically)

**Losses**:
1. denoising_loss: standard diffusion loss
2. proj_loss: REPA alignment loss

Total loss = denoising_loss + proj_loss * proj_coeff

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/content/REPA/train.py',
    '--exp-name',            'repa-sits8-dinov2-imagenet256',
    '--output-dir',          '/content/results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '1000',
    '--encoder-depth',       '8',
    '--enc-type',            'dinov2-vit-b',
    '--data-dir',            '/content/data/imagenet256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '15000',
    '--checkpointing-steps', '15000',
     '--teacher-ckpt',       '/content/drive/MyDrive/last.pt',
    '--sampling-steps',      '99999999',
]

process = subprocess.Popen(
    cmd, cwd='/content/REPA', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pat = r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)'
pbar = tqdm(total=30000, desc='Training DINOv2')
last_step = 0
for line in process.stdout:
    m = re.search(pat, line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
    else:
        stripped = line.strip()
        if stripped:
            print(stripped, flush=True)
pbar.close()
process.wait()
print('Training complete. Checkpoint saved.')

Training DINOv2:   0%|          | 0/30000 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Steps:  37%|███▋      | 5495/15000 [18:55<31:59,  4.95it/s, grad_norm=0.939, loss=1.11, proj_loss=-0.484]


## 8. Save checkpoint to Drive

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/repa_results', exist_ok=True)
shutil.copytree('/content/results', '/content/drive/MyDrive/repa_results', dirs_exist_ok=True)
print('Checkpoint saved to Drive.')

## 9. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()

ckpts = sorted(glob.glob('/content/results/repa-sits8-dinov2-imagenet256/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/content/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '1000',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
]

result = subprocess.run(cmd, cwd='/content/REPA', env=env, text=True)
print('Exit code:', result.returncode)

## 10. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))

data = np.load(npz_files[-1])
imgs = data[data.files[0]]

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'))
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 11. Compute FID score

In [ ]:
import subprocess, glob, numpy as np, os
from PIL import Image
from tqdm import tqdm

subprocess.run(['pip', 'install', '-q', 'torch-fidelity'], check=True)

gen_npz = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated:', gen_npz)

gen_dir = '/content/gen_images'
os.makedirs(gen_dir, exist_ok=True)
data = np.load(gen_npz)
imgs = data[data.files[0]]
for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
    Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

real_dir = '/content/data/imagenet256/images/val'

subprocess.run([
    'fidelity',
    '--gpu', '0',
    '--fid',
    '--input1', gen_dir,
    '--input2', real_dir,
], text=True)

In [ ]:
  import subprocess, glob, numpy as np, os
  from PIL import Image
  from tqdm import tqdm

  gen_npz = sorted(glob.glob('/content/REPA/samples/**/*.npz',
  recursive=True))[-1]
  print('Generated:', gen_npz)

  gen_dir = '/content/gen_images'
  os.makedirs(gen_dir, exist_ok=True)
  data = np.load(gen_npz)
  imgs = data[data.files[0]]
  for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
      Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

  real_dir = '/content/data/imagenet256/images/val'

  result = subprocess.run([
      'fidelity', '--gpu', '0', '--fid',
      '--input1', gen_dir,
      '--input2', real_dir,
  ], capture_output=True, text=True)

  print(result.stdout)
  print(result.stderr)